In [1]:
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

project = os.getenv('GOOGLE_CLOUD_PROJECT')
print(project)  # DEBUG

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",  # safer model
    vertexai=True,
    project=project
)

munna-genai


In [3]:
# lets define basic tools
from langchain_core.tools import tool

# create and register tools

@tool
def add(a: int|float, b: int|float) -> int|float:
    """Adds two numbers

    Args:
        a (int | float): first argument
        b (int | float): second argument

    Returns:
        int|float: a + b
    """
    return a + b

In [4]:

# Make llm aware of this tools
llm_with_tools = llm.bind_tools([add])

In [5]:
question = "What is the result of 4 + 5 ?"

In [6]:

response_without_tools = llm.invoke(question)

In [7]:
# process response
response_without_tools.pretty_print()

================================== Ai Message ==================================

4 + 5 = **9**


In [8]:

response_with_tools = llm_with_tools.invoke(question)

In [9]:

# process response
response_with_tools.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  add (a793b5ea-b7e5-4840-89f6-b5824fad06d7)
 Call ID: a793b5ea-b7e5-4840-89f6-b5824fad06d7
  Args:
    b: 5
    a: 4


In [11]:
# Now lets add more tools
@tool
def subtract(a: int | float, b: int | float) -> int | float:
    """Subtracts second number from first number

    Args:
        a (int | float): first argument
        b (int | float): second argument

    Returns:
        int | float: a - b
    """
    return a - b


@tool
def multiply(a: int | float, b: int | float) -> int | float:
    """Multiplies two numbers

    Args:
        a (int | float): first argument
        b (int | float): second argument

    Returns:
        int | float: a * b
    """
    return a * b


@tool
def divide(a: int | float, b: int | float) -> int | float:
    """Divides first number by second number

    Args:
        a (int | float): first argument
        b (int | float): second argument

    Returns:
        int | float: a / b

    Raises:
        ValueError: If b is 0
    """
    if b == 0:
        raise ValueError("Division by zero is not allowed")
    return a / b


@tool
def modulus(a: int | float, b: int | float) -> int | float:
    """Finds remainder when first number is divided by second number

    Args:
        a (int | float): first argument
        b (int | float): second argument

    Returns:
        int | float: a % b

    Raises:
        ValueError: If b is 0
    """
    if b == 0:
        raise ValueError("Modulus by zero is not allowed")
    return a % b

In [12]:
# lets bind multiple tools to llm

llm_with_tools = llm.bind_tools([add, subtract, multiply, divide, modulus])

In [13]:
question = """
I have purchase a mobile phone at 100000 rupees
I got 15% discount. what i will end up paying
"""

In [15]:
response_with_tools = llm_with_tools.invoke(question)

In [16]:
response_with_tools.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  multiply (bfd747c8-c9f3-46f8-b59b-e12bb3af9c20)
 Call ID: bfd747c8-c9f3-46f8-b59b-e12bb3af9c20
  Args:
    a: 100000
    b: 0.15


In [17]:
response_with_tools.content_blocks

[{'type': 'tool_call',
  'id': 'bfd747c8-c9f3-46f8-b59b-e12bb3af9c20',
  'name': 'multiply',
  'args': {'a': 100000, 'b': 0.15}}]

In [18]:
for block in response_with_tools.content_blocks:
    if block['type'] == 'tool_call':
        if block['name'] == 'multiply':
            result = multiply.invoke(block['args'])
result

15000.0

In [19]:
llm_with_tools.invoke("What is capital of France?").pretty_print()

================================== Ai Message ==================================

[{'type': 'text', 'text': 'I do not have access to that information. I can perform mathematical operations like addition, subtraction, multiplication, division, and modulus.', 'extras': {'signature': 'CowCAY89a1+F7icbamL9fYTmIC85jjw/2lbkZ2zpAFTOJbzo7P9J6fk16au4m6MOGcRBZZnbr1s34dmfD/Ko0GRpsMbsfBQT2L/3lCyX/gS9WJ9WesJ9a4zkxrz/Zla1JduUe3Pkp7RXKP4hmAbD+cYj+Ie+gYydFjalOXU7cL7lIx1wgGEA6Spg68goZ+Jb3sxruobAUCQjzGtGrZMCCBZJEaNtwJfxlD8LOZMwBiJl1RQD4uawddIHgzAx7f5Yi0hf3AvvjRhwgDQfwnEakOto7CM0r7cjC2cvxm936UPC2+pVtgpjRMMZ5K4mmbRbtMq2wmV6MJmE940YkwQ8PpdGWyb1K9TpxUVKCNkxKQ=='}}]


In [20]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[add, multiply, subtract, divide, modulus]
)

In [21]:
from langchain_core.messages import HumanMessage
messages = [HumanMessage("what is 2 + 2 ?")]

In [22]:
response = agent.invoke({
    "messages": messages
})

In [23]:
response['messages']

[HumanMessage(content='what is 2 + 2 ?', additional_kwargs={}, response_metadata={}, id='14635e77-2813-48b5-bef2-eeb0e7952b37'),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'add', 'arguments': '{"a": 2, "b": 2}'}, '__gemini_function_call_thought_signatures__': {'8ed350f9-0fc4-45d7-a2cb-391b908a28c0': 'CtACAY89a1+TuFhx4snwoQ0CV23LZRfnN5o5D0LNvhNotka8OVdm6GJ6mCD50rE1CmzSWARKra0ZbmBEP7aRPoWTV1ZTZ0ZyoOMyS/miz+9UfQzrBAZskZan9W1YPnhChKOx+2IuotyZnWY36sjC2ylJOFBNCS8TgejHTkseTKqzxrC/ysyyLMdLM6OmFvDH/7tFO8Sq6Xd71CMLAPi0BnRrDR39guGryXomGE9Ypa8Yy4FdurvhcWJLFcYwDR0EcDdwV9VdL5aWb0RNzh5dF7zlBvyzOai8RRPz1nvhRlp3TGfsPX4y5Js5vUdXiiUz42HfOsffdO1SBSCwFmkc2HZwDi5Zb9YNof0imaLNLXEpa5ewrB0NIf6ewf9bR0ibcNKMya5F1IjWMFeeRJCRBpWavdipKK9LaZwsWDTZUtja/gXTeangjGWNZJEwEHvKfIvQ'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d542b-4384-7d61-931a-447883ad4e52-0', tool_calls=[{'name': 'add', 

In [24]:
response['messages'][-1].content

'2 + 2 = 4.'

In [25]:
from langchain_core.messages import HumanMessage, BaseMessage
def ask_question_to_agent(question:str, verbose:bool = True) -> BaseMessage:
    messages = [HumanMessage(question)]
    response = agent.invoke({
        "messages": messages
    })
    if verbose:
        print(response['messages'])
        print(len(response['messages']))
    return response['messages'][-1]
    

In [26]:
question = """
I have purchase a mobile phone at 100000 rupees
I got 15% discount. what i will end up paying
"""

In [27]:
reply = ask_question_to_agent(question=question)

[HumanMessage(content='\nI have purchase a mobile phone at 100000 rupees\nI got 15% discount. what i will end up paying\n', additional_kwargs={}, response_metadata={}, id='b3085fc7-51f8-445b-874f-b090a9197411'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"b": 100000, "a": 0.15}'}, '__gemini_function_call_thought_signatures__': {'e78ed57d-409c-45c5-95d7-fa47eed3eb4c': 'CtQFAY89a19kfUKOgb9JPHrZc6UmNmEmw5UMJaPSeBdN896CieDqQGzQWux8GP4VZOHuL+s84WpW5Bol6AesWX8X1bnPlpA302RRRTsV4uiV1eLOCfRIvCSoQM81Y+YCNK7nf3MSva2uKauVfvnEjVH6xTms8LvnS2hb6wd7H44Wp/wIQPeJlYFfEfmzhCRKx2UZD28OOp11xsboDbQvLfPlLE/kw30vAS2Tujd9w2YoU2W6N2pQ/te00oIxBd9VZiLLbqTX7uWD7YwO6L7kYZgJVuprWfGHmx9YRcR0UmFol1zUQpyYOs8cueHNXcUX6IT8mx2ZC0l7y7vrMJHdW9Rfstmgzct+ORj3tHj8ZWh0hmQS7ZW0fVJullK9v0urUbQZP15QuV+iwz5VJ31ajnW7dg7lFCAZGWzIAhfmbkBvqAkINJ148Hlvp4th6h+SRyY1PkF6umKY9sPLNwe/+0bUZkCk8Z7QxkYttKDxe3cxKOXmaBfI7w14CaATq/bsUMnNPX3GnBwdxbvfFhRZOPrWT+uMitb/VGpSYNiRaGq79q9ewq97ORvgSYuvWMB/+E

In [28]:
reply.pretty_print()

================================== Ai Message ==================================

You will end up paying 85000 rupees.


In [29]:
question = """
I have taken 100000 rupees on loan from a friend at 1.5% interest rate per month.
Interest type is simple
What i would end up paying if i return the amount in 10 months
"""

In [30]:
reply = ask_question_to_agent(question)

[HumanMessage(content='\nI have taken 100000 rupees on loan from a friend at 1.5% interest rate per month.\nInterest type is simple\nWhat i would end up paying if i return the amount in 10 months\n', additional_kwargs={}, response_metadata={}, id='892de85d-741d-44c6-8217-c5713c17c620'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"b": 0.015, "a": 100000}'}, '__gemini_function_call_thought_signatures__': {'bcffe393-90d5-4647-a113-4e80fbd8b9a3': 'CqwHAY89a1+K06VLoKe5OYXAlFwSbGXSqW0wjlcZ3cJ/oYlGiO+vUFIqy+yuWcuwXa8jvnCTcsUpdWBOY9bbWQrcP3j8TyqU+pA0pLLX8aASjvkwnIAP6si2S+gdic+ypOn2G0IbKrnuJhTgCqmC5PgXw7AXk0NK8aDpHPqdr/rx/nDWsw7qXC8nnSszj1+dWW+bpvPkkvfkd30f7JyvAnQYLxVhmw6tOEt0K9CQ/TjaemaYUmp6mhXyD6zYbGxElfrFxufa8Uqct2oaA9KsBujTqZQhvBpB0OPsX+6Uet8wfWz3OHXOWevOAmXIjovgZNMnPOmWUvnYB0CA9+AMDYVrdbPVpFGEQQ7mhFzMS8OvvdB9qj32S4SZ+OTHG+y8m6MkX81h09BBUhFnMfZ0AJcjxFIfPdZUdE6abFrBUTWxHBk6WtDJuC6sYSz1KHj2qrlP0QK25frc5Q6TDsHFvmy5Hwdwy1EH5iLzQYOur62264ZyXVlcf

In [31]:
reply.pretty_print()

================================== Ai Message ==================================

[{'type': 'text', 'text': 'You would end up paying a total of 115000 rupees in 10 months.', 'extras': {'signature': 'CqEDAY89a18G7FRlBQj/JW/aRIlIhJmF3X5VZj2//D/JYgW0wK9laGA/Eagysc+kyzXW+3doRbbBLVne4FFvXboz1EgmWcMSCBB171Q4zws71q6KZFfqh84RVGXkhOErmT0tTMzbXJo7Pd0egWfjAhKS1ws9YYZiKGQ/Al6EyHQ43uckX8LU+1/9MpjME5rwMLiwFbWII0iivi+cXw54LuH2TDzEyviaqwlyhYNvRSby70HftIu/jxNlTVubimGL4m3YWUH7FByWnDaSJbW0PmgZDKsYvDoTy9SQrHiWOwTPJzaCAeQZp+ZGRAh2ZDFZQKD9KySGnR4Tf7EddDCFnUDgjjdWCfAfbOcx2dyuCPegPWn7XjFj/VqrgweTNP3aRx1s5FOXcLBYA1WDFs6nDUePvcqQQ1x/6sGSPEKdDEUGrV2V5q8HG8U5/n8RUc9gkuI9z4+v3hsLs45ka29Bn3dGO/CWaCVvYXbNgK1s2HPl6jyHzpimWNpmMoJMW9bw4yZwwUIYsYyYOngoR4kB24azqodLWzbsJo9a00JKdCL+wAiw'}}]
